In [0]:
%sql
-- STEP 1 CREATE THE RAW BRONZE TABLE
CREATE OR REPLACE TABLE raw_login_data(
    user_id INT ,
    login_time TIMESTAMP,
    status STRING,
    country STRING
);
-- INSERT DATA
INSERT INTO raw_login_data VALUES 
(101, '2026-06-20 10:00:00', 'success', 'India'),
(102, '2026-06-20 10:01:00', 'failed', 'USA'),
(102, '2026-06-20 10:01:00', 'failed', 'USA'),
(103, '2026-06-20 10:02:00', 'failed', 'China'),
(104, '2026-06-20 10:02:00', 'failed', 'China'),
(105, '2026-06-20 10:02:00', 'failed', 'USA'),
(103, '2026-06-20 10:02:00', 'failed', 'India'),
(106, '2026-06-20 10:02:00', 'failed', 'China'),
(106, '2026-06-20 10:02:00', 'failed', 'China');
-- STEP 2: CREATE CLEAN SILVER TABLE
CREATE OR REPLACE TABLE clean_login_data AS
SELECT DISTINCT user_id, login_time, status, country
FROM raw_login_data
WHERE user_id IS NOT NULL;

-- STEP 3: CREATE GOLD REPORT
CREATE OR REPLACE TABLE gold_failed_logins_summary AS
SELECT country, COUNT(*) as total_failures
FROM clean_login_data
WHERE status = 'failed'
GROUP BY country;

-- VIEW RESULTS
SELECT * FROM gold_failed_logins_summary;

In [0]:
import pyspark.sql.functions as F

# 1. READ FROM BRONZE: Load the raw SQL table into a PySpark DataFrame variable
bronze_df = spark.table("raw_login_data")

# 2. TRANSFORM TO SILVER: Filter out nulls and drop duplicates
silver_df = (
    bronze_df
    .filter(bronze_df["user_id"].isNotNull())  # Matches: WHERE user_id IS NOT NULL
    .dropDuplicates(["user_id", "login_time", "status", "country"])  # Matches: SELECT DISTINCT
)

# 3. AGGREGATE TO GOLD: Group by country and calculate the final count
gold_df = (
    silver_df
    .filter(silver_df["status"] == "failed")   # Matches: WHERE status = 'failed'
    .groupBy("country")                        # Matches: GROUP BY country
    .agg(F.count("*").alias("total_failures")) # Matches: COUNT(*) as total_failures
)

# 4. VIEW RESULTS: Show the final data frame on screen
gold_df.show()

In [0]:
# Query: Total login attempts from India
india_logins = (
    spark.table("raw_login_data")
    .filter("country = 'India'")
    .count()
)

print(f"Total login attempts from India: {india_logins}")

In [0]:
# Group login attempts by status
status_summary = (
    spark.table("raw_login_data")
    .groupBy("status")
    .count()
    .orderBy("status")
)

display(status_summary)